In [ ]:
# this script is intended to split the entire student population in the data into tracks

In [ ]:
import pandas as pd
import os
import numpy as np
import gc

In [ ]:
os.chdir(r'H:\Hekmat')

In [ ]:
df = pd.read_pickle('selected_data_v2.pk1')

In [ ]:
# run this if you want to get an idea of the kind of data in each non numeric variable

categorical_columns = df.select_dtypes(include=['object', 'category']).columns

print(categorical_columns.tolist())

for col in categorical_columns:
    print(f"Column: {col}")
    print(f"Dtype: {df[col].dtype}\n")
    
    print(f"frequency table for: {col}\n")
    freq_table = df[col].value_counts(dropna=False).reset_index()
    freq_table.columns = [col, 'count']
    freq_table['percentage'] = (freq_table['count'] / len(df)) * 100

    print (freq_table)
    print ("\n" + "-"*40 + "\n") 

In [ ]:
# Some variables have white space entries, replace with NaN:
for col in df.columns:
    if df[col].dtype.kind in {'O','U'}: # object (string-like) columns
        df[col] = df[col].replace("", np.nan).infer_objects(copy=False)
    elif isinstance(df[col].dtype, pd.CategoricalDtype): # categorical columsn
        if "" in df[col].cat.categories:
            df[col] = df[col].cat.remove_categories("")
        df[col] = df[col].replace("", np.nan).infer_objects(copy=False)

In [ ]:
# run this to check removal of white space entries (and check that cateogircal variables dtype is maintained)

categorical_columns = df.select_dtypes(include=['object', 'category']).columns

print(categorical_columns.tolist())

for col in categorical_columns:
    print(f"Column: {col}")
    print(f"Dtype: {df[col].dtype}\n")
    print(f"frequency table for: {col}\n")
    freq_table = df[col].value_counts(dropna=False).reset_index()
    freq_table.columns = [col, 'count']
    freq_table['percentage'] = (freq_table['count'] / len(df)) * 100

    print (freq_table)
    print ("\n" + "-"*40 + "\n") 

In [ ]:
# CREATE separate dataframes for VWO, Havo, VMBO theoretisch/gemeng, and VMBO kader/basisberoep
# this is based on ONDERWIJSSOORTVODIPL_ variable to get an "ever attempted" inclusion criteria

onderwijs_columns = [col for col in df.columns if "ONDERWIJSSOORTVODIPL_" in col]
matching_rinpersonen = set()
for col in onderwijs_columns:
    matching_rinpersonen.update(df.loc[df[col] == "Vwo", "RINPERSOON"])


In [ ]:
vwo_df = df[df["RINPERSOON"].isin(matching_rinpersonen)]

In [ ]:
vwo_df.shape

In [ ]:
vwo_df.to_pickle('vwo_df.pk1')

In [ ]:
del(vwo_df)
gc.collect()

In [ ]:
matching_rinpersonen_havo = set()

for col in onderwijs_columns:
    matching_rinpersonen_havo.update(df.loc[df[col] == "Havo", "RINPERSOON"])

In [ ]:
havo_df = df[df["RINPERSOON"].isin(matching_rinpersonen_havo)]

In [ ]:
havo_df.shape

In [ ]:
havo_df.to_pickle('havo_df.pk1')

In [ ]:
del(havo_df)
gc.collect()

In [ ]:
matching_rinpersonen_vmbotg = set()

for col in onderwijs_columns:
    matching_rinpersonen_vmbotg.update(df.loc[df[col].isin(["Vmbo-gemengd", "Vmbo-theoretisch"]), "RINPERSOON"])

In [ ]:
vmbo_tg_df = df[df["RINPERSOON"].isin(matching_rinpersonen_vmbotg)]

In [ ]:
vmbo_tg_df.shape

In [ ]:
vmbo_tg_df.to_pickle('vmbo_tg_df.pk1')

In [ ]:
matching_rinpersonen_vmbobk = set()

for col in onderwijs_columns:
    matching_rinpersonen_vmbobk.update(df.loc[df[col].isin(["Vmbo-kaderberoeps", "Vmbo-basisberoeps"]), "RINPERSOON"])

vmbo_bk_df = df[df["RINPERSOON"].isin(matching_rinpersonen_vmbobk)]
vmbo_bk_df.shape

In [ ]:
vmbo_bk_df.to_pickle('vmbo_bk_df.pk1')